# Wetter-Agent mit PydanticAI und Ollama

Vor dem Start: *Laufzeit > Laufzeittyp ändern > T4 GPU*. Nach jedem Neustart der Laufzeit alle Zellen der Reihe nach ausführen.

## Setup

In [ ]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q "pydantic-ai-slim[openai]" httpx

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (588 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video grou

In [ ]:
import subprocess, time

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

In [ ]:
!ollama pull qwen3:8b
!ollama list
!curl -s localhost:11434
!nvidia-smi --query-gpu=name,memory.total --format=csv


NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    
Ollama is runningname, memory.total [MiB]
Tesla T4, 15360 MiB


### __Install Packages and set parameters__

In [ ]:
"""Wetter-Agent mit PydanticAI, Open-Meteo und einem lokalen Ollama-Modell."""

from typing import Annotated

import httpx
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider

GEO_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# WMO-Wettercodes, damit das Modell keine Zahlencodes deuten muss
WMO = {
    0: "klar", 1: "überwiegend klar", 2: "teils bewölkt", 3: "bedeckt",
    45: "Nebel", 48: "Reifnebel",
    51: "leichter Nieselregen", 53: "Nieselregen", 55: "starker Nieselregen",
    61: "leichter Regen", 63: "Regen", 65: "starker Regen",
    71: "leichter Schneefall", 73: "Schneefall", 75: "starker Schneefall",
    80: "leichte Schauer", 81: "Schauer", 82: "heftige Schauer",
    95: "Gewitter", 96: "Gewitter mit Hagel", 99: "Gewitter mit starkem Hagel",
}

Latitude = Annotated[float, Field(ge=-90, le=90)]
Longitude = Annotated[float, Field(ge=-180, le=180)]
Days = Annotated[int, Field(ge=1, le=7)]





### __Create Data Structure and implement Agent__

In [ ]:
# --- Rohdaten der Geocoding-API (Systemgrenze 1: externe API) ---

class Location(BaseModel):
    name: str
    latitude: float
    longitude: float
    country: str | None = None
    admin1: str | None = None  # Bundesland, Region o.ä.


# --- Strukturierte Ausgabe des Agents (Systemgrenze 2: LLM-Output) ---

class DayForecast(BaseModel):
    date: str
    t_min: float
    t_max: float
    precipitation_mm: float
    conditions: str


class WeatherReport(BaseModel):
    location: str
    current_temperature: float
    days: list[DayForecast] = Field(min_length=1, max_length=7)
    summary: str = Field(description="Zwei bis drei Sätze Einschätzung auf Deutsch")

### __Implement the Agent__


In [ ]:
# --- Agent ---

model = OllamaModel(
    "qwen3:8b",
    provider=OllamaProvider(base_url="http://localhost:11434/v1"),
)

agent = Agent(
    model,
    deps_type=set[tuple[float, float]],  # Koordinaten, die geocode wirklich geliefert hat
    output_type=WeatherReport,
    retries=2,
    instructions=(
        "Du bist ein Wetter-Assistent. "
        "Ermittle zuerst mit geocode die Koordinaten des Ortes, "
        "dann hole mit get_forecast die Vorhersage für genau diese Koordinaten. "
        "Erfinde keine Werte, übernimm Zahlen aus den Tool-Ergebnissen. "
        "Antworte auf Deutsch. /no_think"
    ),
)


###  __Obtain the forecast__

In [ ]:
def _key(lat: float, lon: float) -> tuple[float, float]:
    return (round(lat, 2), round(lon, 2))


@agent.tool
def geocode(ctx: RunContext[set[tuple[float, float]]], city: str) -> list[dict]:
    """Findet Orte zu einem Namen und liefert deren Koordinaten."""
    resp = httpx.get(
        GEO_URL,
        params={"name": city, "count": 3, "language": "de", "format": "json"},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json().get("results", [])
    if not results:
        raise ModelRetry(f"Kein Ort namens '{city}' gefunden. Prüfe die Schreibweise.")
    locations = [Location.model_validate(r) for r in results]
    ctx.deps.update(_key(l.latitude, l.longitude) for l in locations)
    return [l.model_dump(exclude_none=True) for l in locations]


@agent.tool
def get_forecast(
    ctx: RunContext[set[tuple[float, float]]],
    latitude: Latitude,
    longitude: Longitude,
    days: Days = 5,
) -> dict:
    """Holt aktuelles Wetter und eine Tagesvorhersage für Koordinaten aus geocode."""
    if _key(latitude, longitude) not in ctx.deps:
        raise ModelRetry(
            "Diese Koordinaten stammen nicht aus geocode. "
            "Rufe zuerst geocode auf und verwende exakt die gelieferten Werte."
        )
    resp = httpx.get(
        FORECAST_URL,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,weather_code,wind_speed_10m",
            "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum,weather_code",
            "timezone": "auto",
            "forecast_days": days,
        },
        timeout=10,
    )
    resp.raise_for_status()
    data = resp.json()
    cur, daily = data["current"], data["daily"]

    return {
        "current": {
            "temperature": cur["temperature_2m"],
            "conditions": WMO.get(cur["weather_code"], "unbekannt"),
            "wind_kmh": cur["wind_speed_10m"],
        },
        "days": [
            {
                "date": d,
                "t_min": lo,
                "t_max": hi,
                "precipitation_mm": p,
                "conditions": WMO.get(c, "unbekannt"),
            }
            for d, lo, hi, p, c in zip(
                daily["time"],
                daily["temperature_2m_min"],
                daily["temperature_2m_max"],
                daily["precipitation_sum"],
                daily["weather_code"],
            )
        ],
    }

### __Run the agent__

In [ ]:
import time


prompt = "An welchen ab dem 1.Oktober 2026 wird es noch über 30 Grad Celsius werden? "
start = time.perf_counter()
result = await agent.run(prompt, deps=set())
print(f"Dauer: {time.perf_counter() - start:.1f} s\n")

r = result.output
print(f"{r.location}: aktuell {r.current_temperature} °C\n")
for d in r.days:
    print(f"{d.date}: {d.t_min} bis {d.t_max} °C, {d.precipitation_mm} mm, {d.conditions}")
print(f"\n{r.summary}\n")

UnexpectedModelBehavior: Exceeded maximum output retries (2)

## Agent-Loop ansehen

In [ ]:
txt = ""
for msg in result.all_messages():
    thoughts = msg.parts[0].content
    print(thoughts, "\n")
    if type(thoughts) == str:
        txt += thoughts
        txt += "\n"
    else:
        txt += str(thoughts)
        txt += "\n"

Wie ist das Wetter in Düsseldorf in den nächsten 4 Tagen ?  

Okay, the user is asking about the weather in Düsseldorf for the next 4 days. First, I need to figure out the coordinates for Düsseldorf using the geocode function. Once I have the latitude and longitude, I can use get_forecast to retrieve the weather data. The final result should include the current temperature, a summary, and the 4-day forecast. I'll make sure to structure the response in German as specified, using the data from the tools without inventing any numbers. Let me start by calling geocode with the city name Düsseldorf.
 

[{'name': 'Düsseldorf', 'latitude': 51.22319, 'longitude': 6.77927, 'country': 'Deutschland', 'admin1': 'Nordrhein-Westfalen'}, {'name': 'Düsseldorf-Pempelfort', 'latitude': 51.23812, 'longitude': 6.78678, 'country': 'Deutschland', 'admin1': 'Nordrhein-Westfalen'}, {'name': 'Flughafen Düsseldorf', 'latitude': 51.28066, 'longitude': 6.76715, 'country': 'Deutschland', 'admin1': 'Nordrhein-Westfa

In [ ]:
with open("result.txt", "w") as file:
  file.write(txt)